# EnergyPulse: Forecast error analysis

The model beats the baselines overall. This notebook asks *where* it is wrong: which hours, months and days,
and whether it errs in a consistent direction (bias).

Needs `models/random_forest.joblib`, so run `python -m energypulse.forecast` first.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from energypulse.config import DB_PATH, MODEL_PATH, TEST_DAYS
from energypulse.features import FEATURE_COLUMNS, TARGET, make_features
from energypulse.forecast import chronological_split, evaluate, load_hourly, load_model

pd.options.display.float_format = "{:,.3f}".format

## 1. Rebuild the test-year predictions
Same steps as training: features, drop unusable rows, split by time. Then predict the test year with the saved model.
Error sign convention: `error = actual - forecast`, so **positive means we under-predicted**.

In [2]:
hourly = load_hourly(DB_PATH)
model = load_model(MODEL_PATH)

table = make_features(hourly[TARGET]).dropna(subset=[TARGET] + FEATURE_COLUMNS)
train, test = chronological_split(table, TEST_DAYS)

test = test.copy()
test["pred_rf"] = model.predict(test[FEATURE_COLUMNS])
test["pred_naive"] = test["lag_24"]  # the "same hour yesterday" baseline
test["err_rf"] = test[TARGET] - test["pred_rf"]
test["err_naive"] = test[TARGET] - test["pred_naive"]

print(len(test), "test hours")
evaluate(test[TARGET], test["pred_rf"])  # should match the MAE/RMSE printed by forecast.py

7948 test hours


{'mae': 0.4566194802147473,
 'rmse': 0.6321495193267218,
 'mape_pct': 63.179071613508874,
 'smape_pct': 44.02760436800506}

## 2. Forecast vs actual, one winter week and one summer week

In [3]:
def plot_week(start, title):
    s = pd.Timestamp(start)
    w = test.loc[s : s + pd.Timedelta(days=7), [TARGET, "pred_rf", "pred_naive"]]
    w = w.rename(columns={TARGET: "actual", "pred_rf": "random forest", "pred_naive": "yesterday"})
    return px.line(w.reset_index().melt(id_vars="timestamp"),
                   x="timestamp", y="value", color="variable", title=title)

plot_week("2010-02-01", "Winter week: actual vs forecasts (kWh)")

In [4]:
plot_week("2010-07-05", "Summer week: actual vs forecasts (kWh)")

## 3. Which hours of the day are hardest?
Mean absolute error by hour, for the forest and for the yesterday baseline.

In [5]:
abs_err = test[["err_rf", "err_naive"]].abs().rename(columns={"err_rf": "random forest", "err_naive": "yesterday"})
by_hour = abs_err.groupby(test["hour"]).mean()
px.line(by_hour.reset_index().melt(id_vars="hour"), x="hour", y="value", color="variable",
        title="MAE by hour of day (kWh)")

Is the forest biased at certain hours? Positive bars mean it under-predicts at that hour.

In [6]:
bias_by_hour = test["err_rf"].groupby(test["hour"]).mean().rename("mean_error")
px.bar(bias_by_hour.reset_index(), x="hour", y="mean_error", title="Average error by hour (actual - forecast)")

## 4. Which months?  A ratio below 1 means the forest beats the baseline that month

In [7]:
month = pd.Series(test.index.to_period("M").astype(str), index=test.index, name="month")
mae_month = abs_err.groupby(month).mean()
mae_month["ratio"] = mae_month["random forest"] / mae_month["yesterday"]
mae_month

,random forest,yesterday,ratio
month,,,
2009-11,0.462,0.549,0.840
2009-12,0.543,0.718,0.757
2010-01,0.563,0.801,0.702
2010-02,0.529,0.699,0.756
2010-03,0.444,0.580,0.765
2010-04,0.461,0.554,0.832
2010-05,0.442,0.575,0.768
2010-06,0.384,0.509,0.754
2010-07,0.399,0.429,0.931


In [8]:
px.bar(mae_month["ratio"].rename("mae_ratio").reset_index(), x="month", y="mae_ratio",
       title="Forest MAE / baseline MAE by month (below 1 = forest wins)")

## 5. The worst days

In [9]:
day = test.index.normalize()
daily = pd.DataFrame({
    "mae_rf": test["err_rf"].abs().groupby(day).mean(),
    "mae_yesterday": test["err_naive"].abs().groupby(day).mean(),
    "avg_kwh": test[TARGET].groupby(day).mean(),
    "hours": test[TARGET].groupby(day).size(),
})
daily["weekday"] = daily.index.day_name()
daily.nlargest(10, "mae_rf")

,mae_rf,mae_yesterday,avg_kwh,hours,weekday
timestamp,,,,,
2009-12-24,1.289,1.751,2.018,24,Thursday
2010-10-18,1.156,1.250,1.885,24,Monday
2010-02-28,1.103,0.904,0.494,24,Sunday
2010-01-24,0.976,1.135,1.576,24,Sunday
2009-12-20,0.952,0.898,0.795,24,Sunday
2010-01-03,0.946,0.913,1.447,24,Sunday
2010-10-31,0.942,1.078,1.025,24,Sunday
2010-03-08,0.912,1.110,1.776,24,Monday
2010-11-04,0.900,0.946,1.774,24,Thursday


## 6. Where does the error come from?
Is the error spread evenly, or concentrated in a few spiky hours? And does the model under-predict peaks
(typical for models that average, like a Random Forest)?

In [10]:
sorted_err = test["err_rf"].abs().sort_values(ascending=False)
worst = sorted_err.head(int(len(sorted_err) * 0.05))
print(f"The worst 5% of hours account for {worst.sum() / sorted_err.sum():.0%} of total absolute error")

The worst 5% of hours account for 19% of total absolute error


In [11]:
level = pd.qcut(test[TARGET], 5, labels=["lowest 20%", "20-40%", "40-60%", "60-80%", "highest 20%"])
test.groupby(level, observed=True)["err_rf"].agg(mean_error="mean", mae=lambda s: s.abs().mean(), hours="size")

,mean_error,mae,hours
energy_kwh,,,
lowest 20%,-0.339,0.339,1592
20-40%,-0.431,0.441,1587
40-60%,-0.196,0.401,1590
60-80%,0.110,0.373,1589
highest 20%,0.659,0.728,1590


In [12]:
px.histogram(test, x="err_rf", nbins=80, title="Distribution of errors (actual - forecast, kWh)")

In [13]:
px.scatter(test.sample(3000, random_state=0), x=TARGET, y="pred_rf", opacity=0.3,
           title="Actual vs forecast (3,000 random test hours)")

## 7. Findings to carry forward
Hardest hours: 7 to 10 pm (MAE 0.64 to 0.67); easiest 3 to 5 am (about 0.19). The forest beats yesterday at every hour, with the biggest gains at 7 to 9 am and 11 am to 3 pm.
Months: wins 12 of 13; strongest Jan 2010 (ratio 0.70); loses Aug 2010 (1.36), likely a low-consumption, steady month.
Worst days: Dec 24, 2009, Oct 18, 2010, Feb 28, 2010; mostly Sundays and the holiday period.
Peaks: forecasts never exceed 3.3 kWh while actuals reach 5.6.
Worst 5% of hours: 19% of total absolute error.
Ideas: holiday features, an absence signal, per-hour prediction intervals.